# 第1章：DSP基础 - 交互式学习

**版本：** v2.0  
**最后更新：** 2026-05-26

本notebook提供第1章内容的交互式学习体验。你可以修改参数、运行代码、观察结果。

## 学习目标

1. 理解傅里叶变换的基本原理
2. 学会用FFT分析信号频谱
3. 理解位置编码与傅里叶变换的联系
4. 掌握DSP在深度学习中的应用

## 1.1 FFT频谱分析 - 交互式实验

修改下面的参数，观察信号频谱如何变化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# 设置绘图风格
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# 参数配置
SIGNAL_LENGTH = 1000
SAMPLING_RATE = 100  # Hz

print("FFT频谱分析 - 交互式实验")
print("=" * 50)
print(f"信号长度: {SIGNAL_LENGTH}")
print(f"采样率: {SAMPLING_RATE} Hz")
print(f"时间长度: {SIGNAL_LENGTH / SAMPLING_RATE} 秒")

In [ ]:
# 交互式参数调整
freq1_slider = widgets.FloatSlider(value=5, min=1, max=20, step=0.5, description='频率1 (Hz):')
freq2_slider = widgets.FloatSlider(value=10, min=1, max=20, step=0.5, description='频率2 (Hz):')
noise_slider = widgets.FloatSlider(value=0.1, min=0, max=1, step=0.05, description='噪声水平:')

def plot_fft(freq1, freq2, noise_level):
    np.random.seed(42)
    
    # 生成信号
    t = np.arange(SIGNAL_LENGTH) / SAMPLING_RATE
    signal = np.sin(2 * np.pi * freq1 * t) + np.sin(2 * np.pi * freq2 * t)
    signal_noisy = signal + noise_level * np.random.randn(SIGNAL_LENGTH)
    
    # 计算FFT
    fft_result = np.fft.fft(signal_noisy)
    frequencies = np.fft.fftfreq(SIGNAL_LENGTH, 1 / SAMPLING_RATE)
    magnitude = np.abs(fft_result)
    
    # 只取正频率
    positive_freq_idx = frequencies > 0
    frequencies_positive = frequencies[positive_freq_idx]
    magnitude_positive = magnitude[positive_freq_idx]
    
    # 绘图
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # 时域信号
    axes[0].plot(t[:200], signal_noisy[:200], 'b-', linewidth=0.8, label='Noisy Signal')
    axes[0].plot(t[:200], signal[:200], 'r--', linewidth=1, label='Clean Signal')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title('Time Domain Signal (first 2 seconds)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 频域信号
    axes[1].plot(frequencies_positive[:100], magnitude_positive[:100], 'b-', linewidth=1)
    axes[1].axvline(freq1, color='r', linestyle='--', linewidth=2, label=f'Freq1: {freq1} Hz')
    axes[1].axvline(freq2, color='g', linestyle='--', linewidth=2, label=f'Freq2: {freq2} Hz')
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('Magnitude')
    axes[1].set_title('Frequency Domain (FFT)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 创建交互式输出
widgets.interact(plot_fft, freq1=freq1_slider, freq2=freq2_slider, noise_level=noise_slider)

## 1.2 位置编码 - 交互式实验

理解Transformer中的位置编码如何工作。

In [ ]:
def positional_encoding(seq_length, d_model):
    """
    生成Transformer位置编码
    PE(pos, 2i) = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    """
    pe = np.zeros((seq_length, d_model))
    position = np.arange(seq_length).reshape(-1, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    pe[:, 0::2] = np.sin(position * div_term)
    if d_model % 2 == 1:
        pe[:, 1::2] = np.cos(position * div_term[:-1])
    else:
        pe[:, 1::2] = np.cos(position * div_term)
    
    return pe

# 生成位置编码
seq_length = 100
d_model = 64
pe = positional_encoding(seq_length, d_model)

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 热力图
ax = axes[0, 0]
im = ax.imshow(pe.T, aspect='auto', cmap='RdBu_r')
ax.set_xlabel('Position')
ax.set_ylabel('Dimension')
ax.set_title('Positional Encoding Heatmap')
plt.colorbar(im, ax=ax)

# 2. 不同维度的周期性
ax = axes[0, 1]
for dim in [0, 8, 16, 32]:
    ax.plot(pe[:, dim], label=f'Dim {dim}', alpha=0.7)
ax.set_xlabel('Position')
ax.set_ylabel('Value')
ax.set_title('Periodicity of Different Dimensions')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. 位置编码的范数
ax = axes[1, 0]
norms = np.linalg.norm(pe, axis=1)
ax.plot(norms, 'b-', linewidth=2)
ax.set_xlabel('Position')
ax.set_ylabel('Norm')
ax.set_title('Norm of Positional Encoding')
ax.grid(True, alpha=0.3)

# 4. 相邻位置的相似度
ax = axes[1, 1]
similarities = []
for i in range(seq_length - 1):
    sim = np.dot(pe[i], pe[i+1]) / (np.linalg.norm(pe[i]) * np.linalg.norm(pe[i+1]))
    similarities.append(sim)
ax.plot(similarities, 'g-', linewidth=1)
ax.set_xlabel('Position')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Similarity Between Adjacent Positions')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n位置编码统计:")
print(f"序列长度: {seq_length}")
print(f"模型维度: {d_model}")
print(f"范数范围: [{norms.min():.4f}, {norms.max():.4f}]")
print(f"相邻相似度范围: [{min(similarities):.4f}, {max(similarities):.4f}]")

## 关键概念总结

### FFT (快速傅里叶变换)
- **作用**: 将时域信号转换到频域
- **应用**: 信号分析、滤波、特征提取
- **复杂度**: O(n log n)

### 位置编码
- **作用**: 为Transformer编码序列位置信息
- **特点**: 不同频率的正弦波组合
- **优势**: 能自动泛化到更长的序列

### DSP与深度学习的联系
- CNN中的卷积 ≈ DSP中的滤波
- 位置编码 ≈ 傅里叶变换
- RNN的隐状态 ≈ 状态空间模型

## 练习题

1. **修改频率**: 在上面的FFT实验中，尝试改变频率1和频率2，观察频谱如何变化。

2. **增加噪声**: 增加噪声水平，观察FFT如何受到影响。

3. **位置编码分析**: 
   - 为什么不同维度有不同的周期？
   - 位置编码的范数为什么近似恒定？

4. **应用思考**:
   - 如何用FFT检测信号中的异常？
   - 位置编码为什么比学习的位置向量更好？

## 进一步学习

- 阅读: `docs/01_dsp/` 中的详细文档
- 代码: `code/ch01_dsp/` 中的完整实现
- 论文: "Attention Is All You Need" (Vaswani et al., 2017)